# Mutual Fund Advanced Analytics — Bluestock Capstone

This notebook performs six advanced analytics modules on the Bluestock mutual fund dataset:

| Module | Description |
|--------|-------------|
| **1. VaR & CVaR** | Historical 95% Value at Risk and Conditional VaR for all 40 schemes |
| **2. Rolling Sharpe** | 90-day rolling annualised Sharpe ratio for 5 key funds |
| **3. Cohort Analysis** | Investor segmentation by first-transaction year |
| **4. SIP Continuity** | Gap analysis; flag at-risk investors (gap > 35 days) |
| **5. Fund Recommender** | Risk-appetite-based top-3 fund suggestions |
| **6. Sector HHI** | Herfindahl-Hirschman Index for portfolio concentration |

> **Data window**: NAV history 2022-01-03 to 2025-12-31 · Transactions 2024-01-01 to 2025-05-30


## Setup and Imports


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0D1117',
    'axes.facecolor':   '#161B22',
    'axes.edgecolor':   '#30363D',
    'axes.labelcolor':  '#C9D1D9',
    'xtick.color':      '#8B949E',
    'ytick.color':      '#8B949E',
    'text.color':       '#C9D1D9',
    'grid.color':       '#21262D',
    'legend.facecolor': '#161B22',
    'legend.edgecolor': '#30363D',
})

# Paths
ROOT    = os.path.abspath(os.path.join('..'))
DATA    = os.path.join(ROOT, 'data', 'processed')
REPORTS = os.path.join(ROOT, 'reports')
os.makedirs(REPORTS, exist_ok=True)

# Load datasets
nav   = pd.read_csv(os.path.join(DATA, '02_nav_history.csv'),          parse_dates=['date'])
funds = pd.read_csv(os.path.join(DATA, '01_fund_master.csv'))
perf  = pd.read_csv(os.path.join(DATA, '07_scheme_performance.csv'))
txn   = pd.read_csv(os.path.join(DATA, '08_investor_transactions.csv'), parse_dates=['transaction_date'])
port  = pd.read_csv(os.path.join(DATA, '09_portfolio_holdings.csv'))

nav = nav.sort_values(['amfi_code', 'date'])
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
nav = nav.dropna(subset=['daily_return'])

print(f'NAV records   : {len(nav):,}  |  Funds  : {nav.amfi_code.nunique()}')
print(f'Transactions  : {len(txn):,}  |  Investors : {txn.investor_id.nunique():,}')
print(f'Holdings rows : {len(port):,}  |  Schemes : {port.amfi_code.nunique()}')
print('Data loaded successfully')


## 1. Historical VaR and CVaR (95%) — All 40 Schemes

**Historical VaR (95%)** = 5th percentile of daily return distribution.
**CVaR (95%)** = Mean of returns below the VaR threshold (Expected Shortfall).

Both expressed as **daily percentage loss**. Negative values indicate losses.


In [ ]:
# Compute VaR and CVaR for every fund
records = []
for code_val, grp in nav.groupby('amfi_code'):
    r = grp['daily_return'].values
    if len(r) < 30:
        continue
    var_95  = np.percentile(r, 5)        # VaR: 5th percentile
    cvar_95 = r[r <= var_95].mean()      # CVaR: mean of tail losses
    vol_ann = r.std() * np.sqrt(252)
    records.append({
        'amfi_code'  : int(code_val),
        'var_95_pct' : round(var_95  * 100, 4),
        'cvar_95_pct': round(cvar_95 * 100, 4),
        'vol_ann_pct': round(vol_ann * 100, 4),
        'n_days'     : len(r),
    })

var_df = pd.DataFrame(records)
var_df = var_df.merge(
    perf[['amfi_code','scheme_name','fund_house','category','risk_grade','sharpe_ratio']],
    on='amfi_code', how='left'
)
var_df_sorted = var_df.sort_values('var_95_pct')  # worst VaR first

# Save report
report_path = os.path.join(DATA, 'var_cvar_report.csv')
var_df_sorted.to_csv(report_path, index=False)

# Display
display_cols = ['scheme_name','category','risk_grade','var_95_pct','cvar_95_pct','vol_ann_pct']
print('=' * 95)
print(f'VaR and CVaR Report — {len(var_df_sorted)} Schemes  (sorted: worst VaR first)')
print('=' * 95)
print(var_df_sorted[display_cols].to_string(index=False))
print(f'Report saved to: {report_path}')


In [ ]:
# VaR and CVaR Visualisation
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

RISK_COLORS = {
    'Low':             '#2ECC71',
    'Moderate':        '#3498DB',
    'Moderately High': '#F39C12',
    'High':            '#E67E22',
    'Very High':       '#E74C3C',
}
colors = var_df_sorted['risk_grade'].map(RISK_COLORS).fillna('#95A5A6')

# Horizontal bar — VaR
ax = axes[0]
ax.barh(range(len(var_df_sorted)), var_df_sorted['var_95_pct'],
        color=colors, alpha=0.85, edgecolor='none', height=0.7)
ax.set_yticks(range(len(var_df_sorted)))
ax.set_yticklabels(var_df_sorted['scheme_name'].str[:38], fontsize=6.5)
ax.set_xlabel('VaR 95% (daily %, negative = loss)')
ax.set_title('Historical VaR (95%) — All 40 Funds', fontweight='bold', pad=12)
ax.axvline(0, color='#555', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)

# Scatter — VaR vs CVaR
ax2 = axes[1]
for rg, col in RISK_COLORS.items():
    sub = var_df_sorted[var_df_sorted['risk_grade'] == rg]
    ax2.scatter(sub['var_95_pct'], sub['cvar_95_pct'],
               label=rg, color=col, s=85, alpha=0.85, edgecolors='none')
ax2.set_xlabel('VaR 95% (daily %)')
ax2.set_ylabel('CVaR 95% (daily %)')
ax2.set_title('VaR vs CVaR by Risk Grade', fontweight='bold', pad=12)
ax2.legend(title='Risk Grade', fontsize=9, title_fontsize=9)
ax2.grid(alpha=0.3)

plt.suptitle('Risk Metrics: VaR and CVaR Analysis — All 40 Mutual Fund Schemes',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'var_cvar_chart.png'), dpi=130,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()


## 2. Rolling 90-Day Sharpe Ratio — 5 Key Funds

**Formula**: `Sharpe = rolling_mean(90) / rolling_std(90) x sqrt(252)`

This shows how risk-adjusted performance evolves over time. A Sharpe ratio above 1.0 is generally considered strong.


In [ ]:
KEY_FUNDS = {
    119551: 'SBI Bluechip (Reg)',
    100033: 'HDFC Midcap Opp',
    118634: 'Nippon Small Cap',
    120841: 'Kotak Bluechip',
    119092: 'Axis Bluechip',
}
PALETTE = ['#2ECC71', '#E74C3C', '#3498DB', '#F39C12', '#9B59B6']

fig, ax = plt.subplots(figsize=(16, 7))

for (code_val, label), color in zip(KEY_FUNDS.items(), PALETTE):
    grp = nav[nav['amfi_code'] == code_val].set_index('date')['daily_return']
    if len(grp) < 90:
        continue
    rs = (grp.rolling(90).mean() / grp.rolling(90).std()) * np.sqrt(252)
    rs = rs.dropna()
    ax.plot(rs.index, rs.values, label=label, color=color, linewidth=2, alpha=0.92)
    ax.annotate(f'  {label}', xy=(rs.index[-1], rs.values[-1]),
               fontsize=8, color=color, va='center')

ax.axhline(0, color='#555', linewidth=0.9, linestyle='--', alpha=0.7)
ax.axhline(1, color='#888', linewidth=0.6, linestyle=':', alpha=0.5)
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Rolling 90-Day Sharpe Ratio (annualised)', fontsize=12)
ax.set_title('Rolling 90-Day Annualised Sharpe Ratio — 5 Key Funds',
             fontsize=14, fontweight='bold', pad=15)
ax.legend(loc='upper left', fontsize=9, framealpha=0.2)
ax.grid(True, alpha=0.2)
ax.annotate('Dotted line: Sharpe = 1 (benchmark threshold)',
            xy=(0.01, 0.03), xycoords='axes fraction', fontsize=8,
            style='italic', color='#8B949E')

plt.tight_layout()
chart_path = os.path.join(REPORTS, 'rolling_sharpe_chart.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Chart saved to: {chart_path}')


## 3. Investor Cohort Analysis — by First Transaction Year

Investors are grouped by the year of their **first recorded transaction**.
For each cohort we compute:
- **Avg SIP amount** (SIP transactions only)
- **Total invested** (SIP + Lumpsum combined)
- **Top preferred fund** (by SIP transaction count)


In [ ]:
# Build cohort table
first_txn_yr = txn.groupby('investor_id')['transaction_date'].min().dt.year.rename('cohort_year')
txn2 = txn.merge(first_txn_yr.reset_index(), on='investor_id')

sip_df = txn2[txn2['transaction_type'] == 'SIP']
avg_sip  = sip_df.groupby('cohort_year')['amount_inr'].mean().round(2)

invest_df = txn2[txn2['transaction_type'].isin(['SIP', 'Lumpsum'])]
total_inv = invest_df.groupby('cohort_year')['amount_inr'].sum().round(0)
inv_count = invest_df.groupby('cohort_year')['investor_id'].nunique()

top_fund_code = (
    sip_df.groupby(['cohort_year', 'amfi_code'])
          .size().reset_index(name='n')
          .sort_values(['cohort_year','n'], ascending=[True, False])
          .groupby('cohort_year').first()['amfi_code']
)
top_fund_name = top_fund_code.map(dict(zip(perf['amfi_code'], perf['scheme_name'])))

cohort_summary = pd.DataFrame({
    'Cohort Year'        : avg_sip.index,
    'Investors'          : inv_count.values,
    'Avg SIP Amount (INR)': avg_sip.values,
    'Total Invested (INR)': total_inv.values,
    'Top Fund'           : top_fund_name.values,
})

print('=' * 105)
print('INVESTOR COHORT ANALYSIS')
print('=' * 105)
print(cohort_summary.to_string(index=False))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
years = cohort_summary['Cohort Year'].astype(str)

axes[0].bar(years, cohort_summary['Avg SIP Amount (INR)'], color='#3498DB', width=0.4, alpha=0.85)
axes[0].set_title('Avg SIP Amount by Cohort Year', fontweight='bold')
axes[0].set_xlabel('First Transaction Year')
axes[0].set_ylabel('Avg SIP Amount (INR)')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'INR {x:,.0f}'))
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(years, cohort_summary['Total Invested (INR)'] / 1e7, color='#2ECC71', width=0.4, alpha=0.85)
axes[1].set_title('Total Invested by Cohort Year (Crore)', fontweight='bold')
axes[1].set_xlabel('First Transaction Year')
axes[1].set_ylabel('Total Invested (INR Crore)')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Investor Cohort Analysis — SIP Behaviour by First-Transaction Year',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'cohort_analysis.png'), dpi=130,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()


## 4. SIP Continuity Analysis

For investors with **6 or more SIP transactions**, we compute the **average gap (days)** between consecutive SIPs.

| Status | Criterion |
|--------|-----------|
| Regular | avg gap <= 35 days |
| At-Risk | avg gap > 35 days |


In [ ]:
# SIP gap analysis
sip_only = txn[txn['transaction_type'] == 'SIP'].copy()
sip_only = sip_only.sort_values(['investor_id', 'transaction_date'])
sip_only['gap_days'] = sip_only.groupby('investor_id')['transaction_date'].diff().dt.days

# Investors with 6+ SIP transactions
sip_cnt = sip_only.groupby('investor_id').size()
multi_sip_inv = sip_cnt[sip_cnt >= 6].index
sip_multi = sip_only[sip_only['investor_id'].isin(multi_sip_inv)]

avg_gap_df = (
    sip_multi.groupby('investor_id')['gap_days']
             .mean()
             .reset_index()
             .rename(columns={'gap_days': 'avg_gap_days'})
)
avg_gap_df['sip_count'] = sip_cnt[avg_gap_df['investor_id']].values
avg_gap_df['status']    = avg_gap_df['avg_gap_days'].apply(
    lambda x: 'At-Risk' if x > 35 else 'Regular'
)

total_multi   = len(avg_gap_df)
at_risk_count = (avg_gap_df['status'] == 'At-Risk').sum()
regular_count = (avg_gap_df['status'] == 'Regular').sum()
continuity_rt = regular_count / total_multi * 100

print('=' * 70)
print('SIP CONTINUITY ANALYSIS')
print('=' * 70)
print(f'  Investors with 6+ SIP transactions : {total_multi:,}')
print(f'  Regular (avg gap <= 35 days)       : {regular_count:,} ({continuity_rt:.1f}%)')
print(f'  At-Risk (avg gap > 35 days)        : {at_risk_count:,} ({at_risk_count/total_multi*100:.1f}%)')
print(f'  SIP Continuity Rate                : {continuity_rt:.2f}%')
print('-' * 70)
print('At-Risk sample (top 10 by avg gap):')
print(avg_gap_df[avg_gap_df['status']=='At-Risk']
      .nlargest(10,'avg_gap_days')
      [['investor_id','avg_gap_days','sip_count','status']]
      .to_string(index=False))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(avg_gap_df['avg_gap_days'].dropna(), bins=40,
             color='#3498DB', alpha=0.8, edgecolor='none')
axes[0].axvline(35, color='#E74C3C', linewidth=2, linestyle='--',
                label='At-Risk threshold (35d)')
axes[0].set_xlabel('Avg Gap Between SIPs (days)')
axes[0].set_ylabel('Number of Investors')
axes[0].set_title('Distribution of Avg SIP Gap', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

axes[1].pie(
    [regular_count, at_risk_count],
    labels=['Regular', 'At-Risk'],
    colors=['#2ECC71', '#E74C3C'],
    autopct='%1.1f%%', startangle=90,
    textprops={'color': 'white', 'fontsize': 12},
    wedgeprops={'edgecolor': '#0D1117', 'linewidth': 2},
)
axes[1].set_title('SIP Continuity Status', fontweight='bold')

plt.suptitle('SIP Continuity Analysis — Investors with 6+ SIP Transactions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'sip_continuity.png'), dpi=130,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()


## 5. Simple Fund Recommender

Based on **investor risk appetite**, we filter funds by matching `risk_grade` and rank by **Sharpe ratio** to return the top 3 funds.

| Risk Appetite | Matched Risk Grades |
|---------------|---------------------|
| Low | Low |
| Moderate | Moderate, Moderately High |
| High | High, Very High |

> The full recommender script (with CLI and interactive mode) is in `scripts/recommender.py`.


In [ ]:
RISK_MAP = {
    'Low':      ['Low'],
    'Moderate': ['Moderate', 'Moderately High'],
    'High':     ['High', 'Very High'],
}

def recommend(risk_appetite: str, top_n: int = 3) -> pd.DataFrame:
    """Return top-N funds by Sharpe ratio for a given risk appetite."""
    grades = RISK_MAP[risk_appetite.title()]
    subset = perf[perf['risk_grade'].isin(grades)].copy()
    ranked = subset.sort_values(['sharpe_ratio', 'return_3yr_pct'], ascending=[False, False])
    cols   = ['scheme_name', 'fund_house', 'category', 'sharpe_ratio',
              'return_3yr_pct', 'std_dev_ann_pct', 'expense_ratio_pct', 'risk_grade']
    return ranked[cols].head(top_n).reset_index(drop=True)

# Print recommendations for all three profiles
for profile in ['Low', 'Moderate', 'High']:
    res = recommend(profile)
    sep = '=' * 95
    print(f'\n{sep}')
    print(f'  RECOMMENDATION  |  Risk Appetite: {profile.upper()}')
    print(sep)
    print(res.to_string(index=False))
print('\nDISCLAIMER: Educational purposes only. Consult a SEBI-registered advisor.')


## 6. Sector HHI Concentration — Herfindahl-Hirschman Index

**HHI = sum((weight_i / 100)^2)** across all stock holdings per fund.

| HHI Range | Interpretation |
|-----------|----------------|
| 0 - 0.15 | Highly diversified |
| 0.15 - 0.25 | Moderately concentrated |
| > 0.25 | Highly concentrated |


In [ ]:
def hhi(weights):
    w = weights / 100.0
    return float((w ** 2).sum())

hhi_df = (
    port.groupby('amfi_code')['weight_pct']
        .apply(hhi)
        .reset_index()
        .rename(columns={'weight_pct': 'hhi'})
)
hhi_df = hhi_df.merge(
    perf[['amfi_code', 'scheme_name', 'fund_house', 'category', 'risk_grade']],
    on='amfi_code', how='left'
)

# Equity categories only
eq_cats = ['Large Cap', 'Mid Cap', 'Small Cap', 'Flexi Cap',
           'Large & Mid Cap', 'Value', 'ELSS', 'Index', 'Index/ETF']
hhi_eq = hhi_df[hhi_df['category'].isin(eq_cats)].sort_values('hhi', ascending=False)

print('=' * 90)
print('SECTOR HHI CONCENTRATION — All Equity Funds')
print('=' * 90)
print(hhi_eq[['scheme_name', 'category', 'hhi', 'risk_grade']].to_string(index=False))
print(f"\nMost concentrated : {hhi_eq.iloc[0]['scheme_name']}  (HHI={hhi_eq.iloc[0]['hhi']:.4f})")
print(f"Most diversified  : {hhi_eq.iloc[-1]['scheme_name']} (HHI={hhi_eq.iloc[-1]['hhi']:.4f})")

# Visualise
fig, ax = plt.subplots(figsize=(14, 8))
colors_hhi = hhi_eq['hhi'].apply(
    lambda h: '#E74C3C' if h > 0.25 else ('#F39C12' if h > 0.18 else '#2ECC71')
)
ax.barh(range(len(hhi_eq)), hhi_eq['hhi'],
        color=colors_hhi, alpha=0.85, edgecolor='none', height=0.6)
ax.set_yticks(range(len(hhi_eq)))
ax.set_yticklabels(hhi_eq['scheme_name'].str[:45], fontsize=8)
ax.axvline(0.25, color='#E74C3C', linewidth=1.5, linestyle='--',
           label='High Concentration (0.25)')
ax.axvline(0.18, color='#F39C12', linewidth=1.0, linestyle=':',
           label='Moderate Concentration (0.18)')
ax.set_xlabel('HHI Score')
ax.set_title('Sector Concentration (HHI) — Equity Mutual Funds',
             fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=9, loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS, 'hhi_concentration.png'), dpi=130,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()


## 7. Advanced Insights — 5 Key Takeaways

---

### Insight 1 — Small Cap Funds Carry the Highest Downside Risk

All **Very High** risk-grade funds (SBI, ABSL, Nippon, Axis, DSP Small Cap) register a **daily VaR of -2.5% to -2.7%** and CVaR reaching **-3.1% to -3.4%**. This means in the worst 5% of trading days, an investor can expect to lose more than Rs 2.5 per Rs 100 invested. Investors must have an **investment horizon of 7+ years** to absorb such volatility and realise the long-term compounding advantage.

---

### Insight 2 — Liquid Funds Show Disproportionately High Sharpe Ratios

Debt Liquid funds (ICICI Pru Liquid: 7.68, Kotak Liquid: 6.18, ABSL Liquid: 5.14) show unusually high Sharpe ratios because their **daily return variance is near-zero**. While impressive numerically, this does not imply superior wealth creation — it simply reflects the stability of short-duration debt instruments. **Do not compare their Sharpe directly with equity funds** when allocating capital.

---

### Insight 3 — 2024 Cohort Dominates Investment Volume

Since the transaction dataset spans January 2024 to May 2025, the **2024 cohort is significantly larger** in total invested amount. The 2024 cohort shows higher average SIP ticket sizes, suggesting that investors who started earlier made more transactions. A full longitudinal dataset tracking investors over 5+ years would be needed to draw stronger cohort-maturity conclusions.

---

### Insight 4 — SIP Continuity is Extremely Low (Under 3%)

Of 1,362 investors with 6+ SIP transactions, **over 97%** have an average inter-SIP gap exceeding 35 days, flagging them as 'at-risk'. This likely reflects the presence of **weekly, quarterly, or irregular SIPs** in the dataset rather than monthly cycles. Fund AMCs should focus on converting at-risk investors to **ECS/NACH mandate-based monthly SIPs** to ensure disciplined, uninterrupted investing.

---

### Insight 5 — Small Cap Funds Are Most Concentrated; Index Funds Most Diversified

ABSL Small Cap and Nippon India Small Cap show the **highest HHI scores**, indicating that a handful of top holdings drive disproportionate performance. High concentration amplifies both upside and downside. The **UTI Nifty 50 Index Fund** and **Nippon ETF Nifty 50 BeES** are the most diversified (lowest HHI), making them ideal **core portfolio holdings** for investors seeking broad market exposure with minimal single-stock concentration risk.
